In [ ]:
from dotenv import load_dotenv
import os
from ultralytics import YOLO
from label_studio_sdk import LabelStudio
import requests

#Datos Label Studio
load_dotenv()
API_KEY = os.getenv("API_KEY")
LABEL_STUDIO_URL = os.getenv("LABEL_STUDIO_URL")
URL_REFRESH = os.getenv("URL_REFRESH")


#Token temporal
response = requests.post(URL_REFRESH,json={"refresh": API_KEY})
ACCESS_TOKEN = response.json().get("access")


#Modelo
MODEL_PATH ='/home/adrian/IA-Futbol/runs/detect/train/weights/best.pt'
MODEL_NAME = "Train1"
model = YOLO(MODEL_PATH)

#Conectamos con label studio
client = LabelStudio(api_key=API_KEY)

#Conectamos al projecto IA-Futbol
projects_info = client.projects.list()
project = next((p for p in projects_info if p.title == "IA-Futbol"),None)
project = client.projects.get(project.id)

#sincronizar datos
storages = client.import_storage.local.list(project=project.id)
for strg in storages:
    client.import_storage.local.sync(strg.id)

In [ ]:
from PIL import Image
import requests
from tqdm import tqdm
from io import BytesIO

tasks = client.tasks.list(project=project.id)
images = []
for i, task in enumerate(tqdm(tasks)):
   
    url = f'http://localhost:8080{task.data['image']}'
    try:
        request = requests.get(url, headers={'Authorization': f'Bearer {ACCESS_TOKEN}'}, stream=True)
        image = Image.open(request.raw)
    except:
        #Refrescamos token
        response = requests.post(URL_REFRESH,json={"refresh": API_KEY})
        ACCESS_TOKEN = response.json().get("access")
        request = requests.get(url, headers={'Authorization': f'Bearer {ACCESS_TOKEN}'}, stream=True)
        image = Image.open(request.raw)
        
    w,h = image.size
    predictions = predict_yolo(image,w,h)
    client.predictions.create(task=task.id, result=predictions['result'], score=predictions['score'], model_version=predictions['model_version'])

In [1]:
from dotenv import load_dotenv
import os
from ultralytics import YOLO
from label_studio_sdk import LabelStudio
import requests

#Datos Label Studio
load_dotenv()
API_KEY = os.getenv("API_KEY")
LABEL_STUDIO_URL = os.getenv("LABEL_STUDIO_URL")
URL_REFRESH = os.getenv("URL_REFRESH")


#Token temporal
response = requests.post(URL_REFRESH,json={"refresh": API_KEY})
ACCESS_TOKEN = response.json().get("access")


#Modelo
MODEL_PATH ='/home/adrian/IA-Futbol/runs/detect/train/weights/best.pt'
MODEL_NAME = "Train1"
model = YOLO(MODEL_PATH)

#Conectamos con label studio
client = LabelStudio(api_key=API_KEY)

#Conectamos al projecto IA-Futbol
projects_info = client.projects.list()
project = next((p for p in projects_info if p.title == "IA-Futbol"),None)
project = client.projects.get(project.id)

#sincronizar datos
storage = client.import_storage.local.list(project=project.id)[0]
client.import_storage.local.sync(storage.id)


verify:True


LocalFilesImportStorage(id=27, type='localfiles', synchronizable=True, path='/home/adrian/IA-Futbol/Dataset/detection/no_label/images', regex_filter='', use_blob_urls=True, last_sync=datetime.datetime(2025, 7, 9, 9, 58, 48, 217516, tzinfo=TzInfo(UTC)), last_sync_count=0, last_sync_job=None, status='completed', traceback='Traceback (most recent call last):\n  File "/home/adrian/anaconda3/envs/IA-Futbol/lib/python3.12/site-packages/label_studio/io_storages/base_models.py", line 441, in _scan_and_create_links\n    link_objects = self.get_data(key)\n                   ^^^^^^^^^^^^^^^^^^\n  File "/home/adrian/anaconda3/envs/IA-Futbol/lib/python3.12/site-packages/label_studio/io_storages/localfiles/models.py", line 97, in get_data\n    return load_tasks_json(blob, key)\n           ^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File "/home/adrian/anaconda3/envs/IA-Futbol/lib/python3.12/site-packages/label_studio/io_storages/utils.py", line 187, in load_tasks_json\n    return load_tasks_json_func(blob, key)\n 

In [3]:
import requests

# Configura tus valores
BASE_URL = "http://localhost:8080"  # sin barra al final
PROJECT_ID = 7                       # reemplaza con el ID real de tu proyecto

response = requests.post(URL_REFRESH,json={"refresh": API_KEY})
ACCESS_TOKEN = response.json().get("access")

In [ ]:
#exportar
import zipfile
import io
response = requests.post(URL_REFRESH,json={"refresh": API_KEY})
ACCESS_TOKEN = response.json().get("access")

url = f"{BASE_URL}/api/projects/7/export?exportType=YOLO"
headers = {
    "Authorization": f"Bearer {ACCESS_TOKEN}"
}

response = requests.get(url, headers=headers, verify=False)  # verify=False si usas localhost con HTTPS sin certificado válido

if response.status_code == 200:
    with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        # Extraemos todo en la carpeta actual (o especifica otra ruta)
        z.extractall()
else:
    print(f"Error {response.status_code} al descargar la exportación")

Error 500 al descargar la exportación


In [23]:
response = requests.post(URL_REFRESH,json={"refresh": API_KEY})
ACCESS_TOKEN = response.json().get("access")

url = f"{BASE_URL}/api/projects/7/exports/"
headers = {
    "Authorization": f"Bearer {ACCESS_TOKEN}"
}

response = requests.get(url, headers=headers)

if response.status_code == 200:
    data = response.json()
    print("ok")
    print(response.json())
   

ok
[{'title': 'IA-Futbol-at-2025-07-09-10-08', 'id': 7, 'created_by': {'id': 1, 'first_name': '', 'last_name': '', 'email': 'adriansk@ucm.es', 'avatar': None}, 'created_at': '2025-07-09T10:08:20.897049Z', 'finished_at': '2025-07-09T10:08:22.889280Z', 'status': 'completed', 'md5': '716513ccc3e988adb14106bad3505725', 'counters': {'task_number': 3290}, 'converted_formats': []}, {'title': 'IA-Futbol-at-2025-07-09-10-06', 'id': 6, 'created_by': {'id': 1, 'first_name': '', 'last_name': '', 'email': 'adriansk@ucm.es', 'avatar': None}, 'created_at': '2025-07-09T10:06:45.382881Z', 'finished_at': '2025-07-09T10:06:47.361139Z', 'status': 'completed', 'md5': '716513ccc3e988adb14106bad3505725', 'counters': {'task_number': 3290}, 'converted_formats': []}, {'title': 'IA-Futbol-at-2025-07-09-10-05', 'id': 5, 'created_by': {'id': 1, 'first_name': '', 'last_name': '', 'email': 'adriansk@ucm.es', 'avatar': None}, 'created_at': '2025-07-09T10:05:50.936260Z', 'finished_at': '2025-07-09T10:05:53.060203Z', '